# Creating AI without such libraries as Torch or TensorFlow

## Load dataset and STOI

In [ ]:
import numpy as np
import json

data = np.memmap("train.bin", dtype=np.uint32, mode="r")

with open("stoi.json", "r") as f:
    stoi = json.load(f)

print(f"Vocabulary length: {len(stoi)}")
print(f"Tokens length: {len(data)}")

## Split dataset into training and test

In [ ]:
data = data

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [ ]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else test_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

# Streaming batching as there is too much data

In [1]:
import numpy as np
import random
import re
from tokenizers import Tokenizer


file_path = "train.txt"
tokenizer = Tokenizer.from_file("stories_tokenizer.json")

token_buffer = np.array([], dtype=np.int32)

CHUNK_SIZE = 500_000
MIN_BUFFER_TOKENS = 500_000
MAX_BUFFER_TOKENS = 700_000


def clean_text(text):
    text = text.replace("<|endoftext|>", " endoftext ")
    text = text.replace("<|end|>", " endoftext ")
    text = re.sub(r'<\|[^|]*\|>', '', text)
    text = ' '.join(text.split())
    return text


def load_random_chunk():
    with open(file_path, "rb") as f:
        f.seek(0, 2)
        file_size = f.tell()
        pos = random.randint(0, max(1, file_size - CHUNK_SIZE))
        f.seek(pos)
        chunk = f.read(CHUNK_SIZE)

    text = chunk.decode("utf-8", errors="ignore")
    text = clean_text(text)
    tokens = tokenizer.encode(text).ids
    return np.array(tokens, dtype=np.int32)


def refill_buffer():
    global token_buffer

    while len(token_buffer) < MIN_BUFFER_TOKENS:
        new_tokens = load_random_chunk()

        if len(new_tokens) < 100:
            continue

        token_buffer = np.concatenate([token_buffer, new_tokens]) if len(token_buffer) > 0 else new_tokens

        if len(token_buffer) > MAX_BUFFER_TOKENS:
            trim_start = random.randint(0, len(token_buffer) - MAX_BUFFER_TOKENS)
            token_buffer = token_buffer[trim_start:trim_start + MAX_BUFFER_TOKENS]


def get_batch(block_size, batch_size):
    global token_buffer

    refill_buffer()

    needed = block_size * batch_size + 1
    if len(token_buffer) < needed:
        raise ValueError(
            f"Buffer too small ({len(token_buffer)}) for a batch of "
            f"batch_size={batch_size}, block_size={block_size}."
        )

    max_start = len(token_buffer) - block_size - 1

    if random.random() < 0.5:
        starts = np.random.randint(0, max_start, size=batch_size)
    else:
        max_base = max_start - batch_size * block_size
        if max_base <= 0:
            starts = np.random.randint(0, max_start, size=batch_size)
        else:
            base = random.randint(0, max_base)
            starts = [base + i * block_size for i in range(batch_size)]

    x = np.stack([token_buffer[i:i + block_size] for i in starts])
    y = np.stack([token_buffer[i + 1:i + block_size + 1] for i in starts])

    consume_up_to = int(np.max(starts)) + block_size + 1
    token_buffer = token_buffer[consume_up_to:]

    return x, y

## Traing loop

In [2]:
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

def check_model_output(model, prompt, max_tokens):
    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = model.generate(context, max_tokens, 1)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)

In [ ]:
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 512
n_heads = 8
block_layers = 8
block_size = 256

batch_size = 64
vocabulary_size = 24_000

gradient = Adam(
    lr=4e-4,
    warmup_steps=1500,
    min_lr=5e-5,
    device="gpu"
)

model = MiniGPT(vocab_size=vocabulary_size, 
                d_model=d_model, 
                block_size=block_size,
                n_layers=block_layers,
                n_heads=n_heads,
                gradient=gradient, 
                device="gpu", 
                quant=32)

ema_loss = None

for step in range(30_000):
    xb, yb = get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step == 1:
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 100 == 0:
        check_model_output(model, "Tell me a story", 100)
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        model.save("saved_model")

model.save("saved_model")

KeyboardInterrupt: 

## Custom Decoder

In [ ]:
from NoTorchAI.LLM.MiniGPT import MiniGPT


# model = MiniGPT.__new__(MiniGPT)
# model: MiniGPT = model.load("saved_model")

# itos = {i: ch for ch, i in stoi.items()}

prompt = "History "
encoded_text = np.array([stoi[ch] for ch in prompt], dtype=np.uint32)
context = encoded_text.reshape(1, -1)

generated = generate(model, context, 30)

output_text = "".join([itos[int(num)] for num in generated[0]])
print(output_text)

## Hugging Face Decoder

In [3]:
from NoTorchAI.LLM.MiniGPT import MiniGPT
import numpy as np
from tokenizers import Tokenizer


tokenizer = Tokenizer.from_file("stories_tokenizer.json")

model = MiniGPT.__new__(MiniGPT)
model: MiniGPT = model.load("saved_model")


def check_model_output(model, prompt, max_tokens):
    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=cp.uint32).reshape(1, -1)

    generated = model.generate(context, max_tokens, 0.5)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)


check_model_output(model, "Tell me a story about Dasha", 200)

Tell me a story about D asha and things that can be fun . You can tell us stories and stories about them . But you have to listen to me and respect the rules . Do you understand ?" Lily and Ben nodded . They said , " Yes , mom . We understand . We understand . We will be nice and nice . We will be nice and nice ." Mom hugged them back . She said , " I love you , my loves . I love you too . And I love you too . We love you ." Ben and Lily smiled . They said , " Thank you , Mom . We love you too . We are good friends ." <| again . <| end . <| end . <| end <| end . <| end . <| end . <| end . <| end !" <| end . <| end . <| end . <| end . <| end . <| end . <| end . <| end . <| end . <| end ." <| end . <| end . <| end . <| end . <| end . <| end ," Ben said . <| end
